# Exercise: the detective game

**Duration** ~45 min &nbsp;·&nbsp; **Session** Day 2, Python notebooks

A real screening question. You are given 24 images of *C. elegans*, each one a
well from a 384-well plate. In some wells the worms are **alive**, in others
**dead**, and the whole well is one or the other.

**Your job: decide automatically, for a whole image, which it is.**

This is a genuine assay. The worms were infected with *Enterococcus faecalis*;
half the wells were treated with ampicillin and half were left untreated. Live
worms **curl**; dead worms go **rod-like and straight**. A screen for new
antibiotics works by scoring thousands of such wells, which is far too many to
score by eye.

**Data**: `data/bbbc010/`: 12 alive wells (columns 01-12, ampicillin-treated)
and 12 dead wells (columns 13-24, untreated).

In [ ]:
import numpy as np
import pandas as pd
import tifffile
import matplotlib.pyplot as plt

from skimage.measure import regionprops_table
from skimage.color import label2rgb

from course import DATA, show

WORMS = DATA / "bbbc010"


def well_of(path):
    """'A05_worm_labels.tif' -> 'A05'"""
    return path.name.split("_")[0]


def kind_of(well):
    """Columns 01-12 were treated and are alive; 13-24 were not and are dead."""
    return "alive" if int(well[1:]) <= 12 else "dead"


annotations = sorted((WORMS / "gt").glob("*_worm_labels.tif"))
print(len(annotations), "wells")
print(pd.Series([kind_of(well_of(p)) for p in annotations]).value_counts().to_string())

## Task 1: look before you measure

Display one alive well and one dead well, each next to its annotation. Then write
down what you expect to differ.

<details>
<summary>Hint</summary>

The images are `data/bbbc010/images/<well>_gfp.tif`, the annotations
`data/bbbc010/gt/<well>_worm_labels.tif`. `A05` is alive, `A17` is dead.
</details>

In [ ]:
# --- your turn ---
fig, axes = plt.subplots(1, 4, figsize=(19, 4))
for offset, well in enumerate(["A05", "A17"]):
    image = ...     # TODO
    labels = ...    # TODO
    ... display image and annotation ...
plt.show()

**Your answer:** which shape features do you expect to separate curled worms from
straight ones? Name two, and say which direction each should go.
*(edit this cell)*

## Task 2: measure every worm in every well

Build one table with a row per worm, keeping the `well` it came from and its
`kind`. Shape features only.

<details>
<summary>Hint 1, which properties?</summary>

`area`, `perimeter`, `eccentricity`, `solidity`, `extent`,
`axis_major_length`, `axis_minor_length`.
</details>

<details>
<summary>Hint 2: the loop</summary>

For each path in `annotations`: read it, `regionprops_table`, wrap in a
`DataFrame`, add `well` and `kind` columns, and collect. Then `pd.concat`.
</details>

In [ ]:
SHAPE_FEATURES = ("area", "perimeter", "eccentricity", "solidity", "extent",
                  "axis_major_length", "axis_minor_length")

# --- your turn ---
frames = []
for path in annotations:
    ...   # TODO: measure this well, tag it with well and kind

worms = pd.concat(frames, ignore_index=True)
worms["aspect"] = ...   # TODO: major axis over minor axis

print(f"{len(worms)} worms in {worms.well.nunique()} wells")
worms.groupby("kind")[["area", "eccentricity", "solidity", "extent", "aspect"]].median().round(3)

Look at `solidity`: area divided by the area of the convex hull. A straight rod
nearly fills its hull, so solidity is high. A curled worm does not: the hull
spans the curl, and solidity drops.

That is a shape feature with a direct mechanical interpretation, which is exactly
what you want when you have to defend a screening result.

## Task 3: can you classify a single worm?

Score each feature: for the best possible cut-off, what fraction of *individual
worms* would be labeled correctly?

<details>
<summary>Hint</summary>

For a cut-off `v` the rule "alive if feature <= v" has accuracy equal to the mean
of *(fraction of alive worms at or below v)* and *(fraction of dead worms above
v)*. Check the opposite direction too, and try every distinct value.
</details>

In [ ]:
def best_split(values_a, values_b, candidates):
    """Best single-threshold accuracy separating two groups."""
    # --- your turn ---
    best_accuracy, best_value = 0.0, None
    for value in candidates:
        above = ...    # TODO: accuracy of "group A is above value"
        below = ...    # TODO: accuracy of "group A is at or below value"
        ...            # TODO: keep the best
    return best_accuracy, best_value


candidates = ["area", "eccentricity", "solidity", "extent", "aspect"]

print(f"{'feature':16s} {'accuracy':>9s} {'cut-off':>10s}   (per worm)")
print("-" * 50)
for column in candidates:
    alive = worms.loc[worms.kind == "alive", column].to_numpy()
    dead = worms.loc[worms.kind == "dead", column].to_numpy()
    accuracy, value = best_split(alive, dead, np.unique(worms[column]))
    print(f"{column:16s} {accuracy:9.3f} {value:10.3f}")

**Your answer:** the best you can do on a single worm is around 78%. Look at an
overlaid histogram of `solidity` and explain why. *(edit this cell)*

In [ ]:
plt.figure(figsize=(7, 4))
bins = np.histogram_bin_edges(worms.solidity, bins=30)
for kind in ("alive", "dead"):
    plt.hist(worms.loc[worms.kind == kind, "solidity"], bins=bins, alpha=0.6, label=kind)
plt.xlabel("solidity"); plt.ylabel("worms"); plt.legend()
plt.title("individual worms overlap")
plt.show()

## Task 4: but you do not have to classify a single worm

Re-read the question. You are asked to classify a **well**, and each well
contains a dozen worms.

Some worms in a treated well die anyway, and some untreated worms are still
moving: the phenotype is "mostly alive" or "mostly dead", not "all". Summarising
the whole well averages that variability away.

Build a per-well table: one row per well, each feature replaced by its **median**
across the worms in that well.

<details>
<summary>Hint</summary>

`worms.groupby(["well", "kind"])[candidates].median().reset_index()`.
</details>

In [ ]:
# --- your turn ---
wells = ...   # TODO: one row per well, median of each feature

print(f"{len(wells)} wells")
wells.head()

In [ ]:
plt.figure(figsize=(7, 4))
bins = np.histogram_bin_edges(wells.solidity, bins=15)
for kind in ("alive", "dead"):
    plt.hist(wells.loc[wells.kind == kind, "solidity"], bins=bins, alpha=0.6, label=kind)
plt.xlabel("median solidity of the well"); plt.ylabel("wells"); plt.legend()
plt.title("wells separate cleanly")
plt.show()

## Task 5: score the well-level rule

Repeat task 3's scoring on the per-well table.

<details>
<summary>Hint</summary>

The same `best_split`, with `wells` in place of `worms`.
</details>

In [ ]:
# --- your turn ---
for column in candidates:
    ...   # TODO: same scoring, on `wells` instead of `worms`

**Your answer:** three features now classify every single well correctly, from
worm-level features that were only ~78% accurate. Explain how averaging achieved
that. *(edit this cell)*

## Task 6: do not believe your own 100%

That perfect score has a problem: the cut-off was chosen **using the same 24
wells it was then tested on**. Of course it fits them.

The fix is to hold data out. **Leave-one-out cross-validation** does this to the
limit: for each well in turn, choose the cut-off from the *other 23*, then
classify the held-out one.

<details>
<summary>Hint 1: the loop</summary>

For each index `i`: `train = wells.drop(i)`, `test = wells.iloc[i]`. Fit the
cut-off on `train` with `best_split`, then predict `test`.
</details>

<details>
<summary>Hint 2: the rule direction</summary>

Alive worms curl, so they have the **lower** solidity: predict `alive` if the
held-out well is at or below the cut-off.
</details>

In [ ]:
FEATURE = "solidity"

# --- your turn ---
correct = 0
for i in range(len(wells)):
    train = wells.drop(i)
    test = wells.iloc[i]
    _, cutoff = ...        # TODO: fit the cut-off on train only
    prediction = ...       # TODO: classify the held-out well
    ...                    # TODO: count it

print(f"leave-one-out accuracy: {correct}/{len(wells)}")

**Your answer:** the honest accuracy is a little below the 100% from task 5. Which
number would you put in a paper, and why? *(edit this cell)*

```{note}
The gap between those two numbers is small here, because the separation is
genuinely large. It will not always be small. Any accuracy measured on the data
used to choose the model is optimistic, and the more choices you made: which
feature, which summary statistic, which cut-off: the more optimistic it gets.

Hold data out. With 24 wells, leave-one-out is cheap; with a real screen you
would set aside a proper validation plate.
```

## If you finish early

Everything above used the expert annotations. Segment the worms yourself: 
threshold `<well>_gfp.tif`, clean up, label, and rebuild the table from your own
masks.

Then rerun task 6. The question that matters for a real screen is not whether the
features separate perfect annotations, but whether they still separate what your
pipeline actually produces.